In [5]:
import pandas as pd
import numpy as np

print("Début du chargement des fichiers d'Olist...")

# 1. CHARGEMENT DES FICHIERS DONT ON A BESOIN
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
translations = pd.read_csv('product_category_name_translation.csv')

print("Tous les fichiers ont été chargés avec succès !")

# 2. TRADUCTION DES CATÉGORIES EN ANGLAIS
# On remplace les noms portugais par les noms anglais pour que ce soit plus lisible
products = products.merge(translations, on='product_category_name', how='left')
# Si un produit n'a pas de traduction, on garde son nom de base, sinon on met 'autres'
products['product_category'] = products['product_category_name_english'].fillna(products['product_category_name']).fillna('other')

# 3. FUSION (MERGE) DES TABLES
# On relie les morceaux ensemble grâce aux clés uniques (IDs)
df = orders.merge(items, on='order_id', how='inner')
df = df.merge(customers, on='customer_id', how='inner')
df = df.merge(products, on='product_id', how='inner')

# 4. NETTOYAGE DES DONNÉES
# A. On ne garde que les commandes livrées sans encombre
df = df[df['order_status'] == 'delivered']

# B. Conversion des dates au format Temporel (indispensable pour prédire le stock futur)
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

# C. Calcul du montant réel de la transaction (Prix de l'objet + Frais de livraison)
df['total_price'] = df['price'] + df['freight_value']

# D. Dans Olist, chaque ligne de la table 'items' correspond à 1 unité vendue
df['quantity'] = 1

# 5. SÉLECTION DES COLONNES ESSENTIELLES
# On filtre pour ne pas s'encombrer avec des colonnes inutiles
columns_to_keep = [
    'customer_unique_id',        # Pour grouper tes clients (Clustering)
    'customer_state',            # Région / État du Brésil (Clustering géo)
    'order_id',                  # Identifiant de la commande
    'order_purchase_timestamp',     # Date exacte de l'achat (Prédiction de stock)
    'product_id',                # Identifiant du produit
    'product_category',          # Catégorie traduite (Ex: health_beauty, watches_gifts...)
    'quantity',                  # Quantité (toujours 1 par ligne ici)
    'price',                     # Prix unitaire
    'total_price',               # Prix total avec livraison
    'freight_value'
]

final_df = df[columns_to_keep]

print("\n--- NETTOYAGE TERMINÉ ---")
print(f"Nombre total de lignes exploitables : {final_df.shape[0]}")
print("\nAperçu des 3 premières lignes de ton dataset propre :")
print(final_df.head(3))

Début du chargement des fichiers d'Olist...
Tous les fichiers ont été chargés avec succès !

--- NETTOYAGE TERMINÉ ---
Nombre total de lignes exploitables : 110197

Aperçu des 3 premières lignes de ton dataset propre :
                 customer_unique_id customer_state  \
0  7c396fd4830fd04220f754e42b4e5bff             SP   
1  af07308b275d755c9edb36a90c618231             BA   
2  3a653a41f6f9fc3d2a113cf8398680e8             GO   

                           order_id order_purchase_timestamp  \
0  e481f51cbdc54678b7cc49136f2d6af7      2017-10-02 10:56:33   
1  53cdb2fc8bc7dce0b6741e2150273451      2018-07-24 20:41:37   
2  47770eb9100c2d0c44946d9cf07ec65d      2018-08-08 08:38:49   

                         product_id product_category  quantity   price  \
0  87285b34884572647811a353c7ac498a       housewares         1   29.99   
1  595fac2a385ac33a80bd5114aec74eb8        perfumery         1  118.70   
2  aa4383b373c6aca5d8797843e5594415             auto         1  159.90   

   total_p

In [6]:
import pandas as pd
import numpy as np

# On s'assure que la colonne date est bien au bon format
final_df['order_purchase_timestamp'] = pd.to_datetime(final_df['order_purchase_timestamp'])

print("Création du Dataset 1 : Clustering RFM...")

# Date de référence pour calculer la récence (le jour d'après la toute dernière commande du dataset)
snapshot_date = final_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# Agrégation par client unique
dataset_clustering = final_df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days, # Récence
    'order_id': 'nunique',                                               # Fréquence
    'total_price': 'sum',                                                # Montant
    'customer_state': 'first'                                            # Localisation géo
}).reset_index()

# Renommer les colonnes de manière professionnelle
dataset_clustering.columns = ['customer_unique_id', 'recency', 'frequency', 'monetary_value', 'customer_state']

# Sauvegarde du premier dataset pour ton rendu
dataset_clustering.to_csv('olist_engineered_clustering.csv', index=False)
print(f"-> Dataset Clustering créé : {dataset_clustering.shape[0]} clients uniques segmentés.")


print("\nCréation du Dataset 2 : Prédiction des Stocks...")

# Extraction de la période (Année-Semaine) pour regrouper les ventes par maille hebdomadaire
final_df['year_week'] = final_df['order_purchase_timestamp'].dt.to_period('W').astype(str)

# Agrégation par semaine, catégorie et état logistique
dataset_stocks = final_df.groupby(['year_week', 'product_category', 'customer_state']).agg({
    'quantity': 'sum',         # Volume total vendu (Target pour ton ML)
    'total_price': 'sum'       # Chiffre d'affaires généré sur cette catégorie
}).reset_index()

# Renommer la colonne cible
dataset_stocks.rename(columns={'quantity': 'weekly_sales_volume'}, inplace=True)

# Sauvegarde du deuxième dataset pour ton rendu
dataset_stocks.to_csv('olist_engineered_stocks.csv', index=False)
print(f"-> Dataset Stocks créé : {dataset_stocks.shape[0]} lignes de séries temporelles générées.")

print("\n--- TOUS LES DATASETS ONT ÉTÉ SAUVEGARDÉS AVEC SUCCÈS ---")

C:\Users\maell\AppData\Local\Temp\ipykernel_30564\3628765141.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['order_purchase_timestamp'] = pd.to_datetime(final_df['order_purchase_timestamp'])


Création du Dataset 1 : Clustering RFM...
-> Dataset Clustering créé : 93358 clients uniques segmentés.

Création du Dataset 2 : Prédiction des Stocks...
-> Dataset Stocks créé : 26063 lignes de séries temporelles générées.

--- TOUS LES DATASETS ONT ÉTÉ SAUVEGARDÉS AVEC SUCCÈS ---


C:\Users\maell\AppData\Local\Temp\ipykernel_30564\3628765141.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['year_week'] = final_df['order_purchase_timestamp'].dt.to_period('W').astype(str)


In [7]:
import pandas as pd
import numpy as np

final_df['order_purchase_timestamp'] = pd.to_datetime(final_df['order_purchase_timestamp'])
snapshot_date = final_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

print("Enrichissement du Dataset 1 (Clustering)...")
# On ajoute 'freight_value' (le coût du transport seul) dans l'agrégation
dataset_clustering = final_df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'total_price': 'sum',
    'freight_value': 'mean', # NOUVELLE FEATURE : Coût de transport moyen payé par le client
    'customer_state': 'first'
}).reset_index()

dataset_clustering.columns = ['customer_unique_id', 'recency', 'frequency', 'monetary_value', 'mean_freight', 'customer_state']
dataset_clustering.to_csv('olist_engineered_clustering.csv', index=False)


print("Enrichissement du Dataset 2 (Stocks)...")
final_df['year_week'] = final_df['order_purchase_timestamp'].dt.to_period('W').astype(str)
# NOUVELLE FEATURE : On extrait le mois (1 à 12) pour capturer la saisonnalité
final_df['month'] = final_df['order_purchase_timestamp'].dt.month 

dataset_stocks = final_df.groupby(['year_week', 'month', 'product_category', 'customer_state']).agg({
    'quantity': 'sum',
    'total_price': 'sum'
}).reset_index()

dataset_stocks.rename(columns={'quantity': 'weekly_sales_volume'}, inplace=True)
dataset_stocks.to_csv('olist_engineered_stocks.csv', index=False)

print("\n--- DATASETS MIS À JOUR AVEC LES FEATURES AVANCÉES ---")

C:\Users\maell\AppData\Local\Temp\ipykernel_30564\3196726990.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['order_purchase_timestamp'] = pd.to_datetime(final_df['order_purchase_timestamp'])


Enrichissement du Dataset 1 (Clustering)...
Enrichissement du Dataset 2 (Stocks)...

--- DATASETS MIS À JOUR AVEC LES FEATURES AVANCÉES ---


C:\Users\maell\AppData\Local\Temp\ipykernel_30564\3196726990.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['year_week'] = final_df['order_purchase_timestamp'].dt.to_period('W').astype(str)
C:\Users\maell\AppData\Local\Temp\ipykernel_30564\3196726990.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['month'] = final_df['order_purchase_timestamp'].dt.month


In [8]:
print(final_df.columns.tolist())

['customer_unique_id', 'customer_state', 'order_id', 'order_purchase_timestamp', 'product_id', 'product_category', 'quantity', 'price', 'total_price', 'freight_value', 'year_week', 'month']


In [9]:
import pandas as pd
import numpy as np

# On s'assure que la date de référence est prête
snapshot_date = final_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

print("🚀 Étape 1 : Génération du Dataset de Clustering (Modèle Client)...")
dataset_clustering = final_df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days, # Récence
    'order_id': 'nunique',                                               # Fréquence
    'total_price': 'sum',                                                # Montant Total
    'freight_value': 'mean',                                             # Coût moyen du fret
    'customer_state': 'first'                                            # Région géo
}).reset_index()

# On renomme proprement pour le rendu
dataset_clustering.columns = ['customer_unique_id', 'recency', 'frequency', 'monetary_value', 'mean_freight', 'customer_state']
dataset_clustering.to_csv('olist_engineered_clustering.csv', index=False)
print("✅ Fichier 'olist_engineered_clustering.csv' sauvegardé !")


print("\n📈 Étape 2 : Agrégation et application du SHIFT pour le Dataset de Stocks...")
# Agrégation par semaine, mois, catégorie et région
dataset_stocks = final_df.groupby(['year_week', 'month', 'product_category', 'customer_state']).agg({
    'quantity': 'sum',
    'total_price': 'sum'
}).reset_index()

dataset_stocks.rename(columns={'quantity': 'weekly_sales_volume'}, inplace=True)

# /!\ TRÈS IMPORTANT : On trie chronologiquement par produit et région avant de faire le shift
dataset_stocks = dataset_stocks.sort_values(by=['product_category', 'customer_state', 'year_week']).reset_index(drop=True)

# On crée le groupe pour appliquer le décalage sans mélanger les produits ou les états
grouped = dataset_stocks.groupby(['product_category', 'customer_state'])

# L'ingénierie des variables historiques (Le fameux décalage !)
dataset_stocks['sales_lag_1'] = grouped['weekly_sales_volume'].shift(1) # Ventes de la semaine d'avant
dataset_stocks['sales_lag_2'] = grouped['weekly_sales_volume'].shift(2) # Ventes d'il y a 2 semaines

# Moyenne mobile des 4 semaines précédentes (La tendance récente)
dataset_stocks['sales_moving_avg_4'] = grouped['weekly_sales_volume'].transform(lambda x: x.shift(1).rolling(window=4, min_periods=1).mean())

# On remplace les NaN (qui apparaissent naturellement au début de l'historique car la semaine d'avant n'existe pas encore) par des 0
dataset_stocks.fillna(0, inplace=True)

# Sauvegarde du second fichier
dataset_stocks.to_csv('olist_engineered_stocks.csv', index=False)
print("✅ Fichier 'olist_engineered_stocks.csv' sauvegardé avec les variables décalées (Lags) !")

print("\n--- 🏁 FIN DU FEATURE ENGINEERING : TOUT EST PRÊT POUR TON RENDU D2 ---")

🚀 Étape 1 : Génération du Dataset de Clustering (Modèle Client)...
✅ Fichier 'olist_engineered_clustering.csv' sauvegardé !

📈 Étape 2 : Agrégation et application du SHIFT pour le Dataset de Stocks...
✅ Fichier 'olist_engineered_stocks.csv' sauvegardé avec les variables décalées (Lags) !

--- 🏁 FIN DU FEATURE ENGINEERING : TOUT EST PRÊT POUR TON RENDU D2 ---


In [11]:
dataset_stocks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27667 entries, 0 to 27666
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   year_week            27667 non-null  object 
 1   month                27667 non-null  int32  
 2   product_category     27667 non-null  object 
 3   customer_state       27667 non-null  object 
 4   weekly_sales_volume  27667 non-null  int64  
 5   total_price          27667 non-null  float64
 6   sales_lag_1          27667 non-null  float64
 7   sales_lag_2          27667 non-null  float64
 8   sales_moving_avg_4   27667 non-null  float64
dtypes: float64(4), int32(1), int64(1), object(3)
memory usage: 1.8+ MB


In [12]:
dataset_clustering.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93358 entries, 0 to 93357
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_unique_id  93358 non-null  object 
 1   recency             93358 non-null  int64  
 2   frequency           93358 non-null  int64  
 3   monetary_value      93358 non-null  float64
 4   mean_freight        93358 non-null  float64
 5   customer_state      93358 non-null  object 
dtypes: float64(2), int64(2), object(2)
memory usage: 4.3+ MB
